# Updating probe and object

In [1]:
import importlib
import vector_ptycho.utils as vpu
importlib.reload(vpu)
from vector_ptycho.utils import *
import numpy as np
import matplotlib.pyplot as plt
import torch
import pickle


In [2]:
RGB_scale = make_vector_color_map(plot=False)

## Define the object, probe, fluence and XMLD constants

In [7]:
H, W = 100, 100
Lx = 10
Ly = 10

# Build theta, phi from meron-antimeron generator (returns torch tensors)
theta, phi, Mx, My, Mz = make_meron_antimeron_theta_phi(
    Nx=W,
    Ny=H,
    Lx=Lx,
    Ly=Ly,
    plot=False,
    save_path=None,
    export_path=None,
    return_torch=True,
    out_device=device,
    cm = RGB_scale,
)

# Physical XMLD/Jones constants
C = torch.tensor(50.0+ 50.0j, dtype=cdtype, device=device)
A1 = torch.tensor(0.5 + 5j, dtype=cdtype, device=device)
A2 = torch.tensor(0.8 + 5j, dtype=cdtype, device=device)

# Build Jones object from theta, phi
neel = NeelObject(C, A1, A2)
J = neel.build_jones(theta, phi)
obj = JonesObject(J)

# Grid for probe definition
x = torch.linspace(-Lx, Lx, H, device=device)
y = torch.linspace(-Ly, Ly, W, device=device)
X, Y = torch.meshgrid(x, y, indexing='ij')

fluence = torch.scalar_tensor(1e3, dtype=torch.float64, device=device)
R=torch.sqrt(X**2+Y**2) #This helps with defining the probe

Diffuser = 0.8*(torch.sin(1*R)+torch.cos((Y*1+X*1)-0.8*(X-0.2))+torch.cos((Y-X)-0.4*(X))) #(torch.sqrt(3*X**2+1.5*Y**2)+torch.pi/3)#*(torch.exp(-0.1*R)) #np.mod(0.1*(torch.sin(150*R)+torch.cos((Y*10+X*30)**2-0.8*(X*75-0.2))+torch.cos((Y*10-X*33)**2-0.4*(X*50-0.2))),1)

P = torch.zeros_like(R)
P = torch.exp(2j*np.pi*Diffuser)*(torch.exp(-0.8*R))


probes = []
# Calculate the Jones vectors of the probes for different Linear polarisation states
pol_angles = [0, 30, 60, 90]
for angle in pol_angles:
    rad = np.deg2rad(angle)
    jones_vec = torch.tensor([np.cos(rad) + 0j, np.sin(rad) + 0j], dtype=cdtype, device=device)
    print(f'Jones vector for {angle} deg linear polarisation: {jones_vec}')
    probes.append(Probe(P, jones_vec, fluence=fluence, normalized=False))

print('Defined {} probes with different linear polarisation states.'.format(len(probes)))


#plot_probe_maps(probes[0].amplitude, Lx, Ly)
print('Probe fluence:', probes[0].fluence)

Jones vector for 0 deg linear polarisation: tensor([1.+0.j, 0.+0.j])
Jones vector for 30 deg linear polarisation: tensor([0.8660+0.j, 0.5000+0.j])
Jones vector for 60 deg linear polarisation: tensor([0.5000+0.j, 0.8660+0.j])
Jones vector for 90 deg linear polarisation: tensor([6.1232e-17+0.j, 1.0000e+00+0.j])
Defined 4 probes with different linear polarisation states.
Probe fluence: tensor(1000., dtype=torch.float64)


# Define the scan trajectory, simulate the detector

In [ ]:
xshiftvec = torch.linspace(-5, 5,10)
yshiftvec = torch.linspace(-5, 5,10)

# Shifting the text so that we can plot on the data
dx = (xshiftvec[1] - xshiftvec[0]) / 10
dy = (yshiftvec[1] - yshiftvec[0]) / 10

# Lab-coordinate scan grid used for plotting
xpos_lab, ypos_lab = torch.meshgrid(xshiftvec, yshiftvec, indexing="ij")
positions = torch.stack([ypos_lab.flatten(), xpos_lab.flatten()], dim=1)

# Convert lab coordinates into discrete pixel shifts for torch.roll(row, col)
pixel_size_y = (2 * Ly) / (H - 1)
pixel_size_x = (2 * Lx) / (W - 1)
positions_idx = torch.stack([
    torch.round(positions[:, 0] / pixel_size_y),
    torch.round(positions[:, 1] / pixel_size_x),
], dim=1).to(torch.int64)

scan = ScanTrajectory(positions_idx)

# Forward model
model = ForwardModel(obj, Propagator(), Detector(add_poisson_noise=True))
I_sim = model.simulate_all(probes, scan)
print('Simulated data shape:', I_sim.shape)  # (N_probes, N_positions, H, W)

## Reconstruct object and probe with batches

In [ ]:
# I_meas should be your simulated or measured data tensor
# shape: (N_probes, N_scan, Hdet, Wdet)
initial_object = 'random'
initial_probe = 'gaussian'
checkpoint_path = "checkpoint_initial.pt"
load_from_checkpoint = False

if load_from_checkpoint==True and os.path.exists(checkpoint_path)==True:
    recon = Reconstruction.load(
        checkpoint_path,
        scan=scan,
        pol_angles=pol_angles,
        I_meas=I_sim,
        fluence=fluence,
        RGB_scale=RGB_scale,
        Lx=Lx,
        Ly=Ly,
        device=device,
    )
    print(f"Loaded checkpoint from {checkpoint_path}")
else:
    recon = Reconstruction.from_initial_guess(
        scan=scan,
        pol_angles=pol_angles,
        I_meas=I_sim,
        H=H,
        W=W,
        Lx=Lx,
        Ly=Ly,
        fluence=fluence,
        RGB_scale=RGB_scale,
        C=C,
        A1=A1,
        A2=A2,
        device=device,
        initial_object=initial_object,
        initial_probe=initial_probe
    )
    recon.save(checkpoint_path)
    print(f"Created initial checkpoint at {checkpoint_path}")

plot_probe_maps(recon.probe_amplitude.detach().cpu().numpy(), Lx, Ly)
plot_theta_phi_maps(
    recon.theta.detach().cpu().numpy(),
    recon.phi.detach().cpu().numpy(),
    Lx,
    Ly,
    theta_cmap='magma',
    phi_cmap=RGB_scale,
    label_axes=True,
)

In [ ]:

# Set optimizer and learning rates explicitly before running.
optimizer = recon.initialize_optimizer(
    lr_theta=1e-2,
    lr_phi=1e-2,
    lr_object=1e-5,
    lr_probe=1e-2,
)

# Explicit usage with no notebook-global fallback inside Reconstruction:
# recon = Reconstruction.load(
#     "checkpoint.pt",
#     scan=scan,
#     pol_angles=pol_angles,
#     I_meas=I_sim,
#     fluence=fluence,
#     RGB_scale=RGB_scale,
#     Lx=Lx,
#     Ly=Ly,
#     device=device,
# )
# recon.initialize_optimizer(lr_theta=1e-2, lr_phi=1e-2, lr_object=1e-5, lr_probe=1e-2)
# result = recon.run()
result = recon.run(
    num_iterations=500,
    batch_size=5,
    checkpoint_path=checkpoint_path,
    checkpoint_every=100,
    optimizer=optimizer,
)

theta_rec = result["theta"]
phi_rec = result["phi"]
probe_amplitude_rec = result["probe_amplitude"]
C_rec = result["C"]
A1_rec = result["A1"]
A2_rec = result["A2"]
loss_history = result["loss_history"]